# Step 4: Patient-Level Dataset Preparation

**Input:** Labeled PPG data from Step 3

This notebook prepares the final patient-level CSV files for the two dataset tasks.

## Outputs (Two Independent Classification Tasks)
1. **Strict CAD vs Non-CAD:** Unresolved rows removed before final binary cohort construction.
2. **Implemented CVD vs Non-CVD:** CVD evidence defined by ICD-9 prefixes 410-414, 420-429, and 440-448.

## Dataset Features
- Numeric sex encoding (`GENDER_NUM`)
- Comorbidity flags: `is_diabetic`, `has_high_cholesterol`, `is_obese`
- One retained waveform record per `SUBJECT_ID`
- Six one-minute PPG windows from minutes 3-8

**Output:** Final patient-level datasets for CAD and CVD analysis.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CAD_STRICT = INTERMEDIATE_DIR / "PPG_cad_flagged_timeaware_strict.csv"
INPUT_CVD = INTERMEDIATE_DIR / "PPG_cvd_flagged_timeaware.csv"

OUTPUT_CAD_PATIENT_FULL = DATA_DIR / "patient_level_CAD_vs_NonCAD_strict.csv"
OUTPUT_CAD_PATIENT_CASE = INTERMEDIATE_DIR / "patient_level_CAD_only_strict.csv"
OUTPUT_CAD_PATIENT_CONTROL = INTERMEDIATE_DIR / "patient_level_NonCAD_only_strict.csv"

OUTPUT_CVD_PATIENT_FULL = DATA_DIR / "patient_level_CVD_vs_NonCVD.csv"
OUTPUT_CVD_PATIENT_CASE = INTERMEDIATE_DIR / "patient_level_CVD_only.csv"
OUTPUT_CVD_PATIENT_CONTROL = INTERMEDIATE_DIR / "patient_level_NonCVD_only.csv"

## **Import Libraries and Configure Output Paths**

In [ ]:
# ----------------------------
# Load and basic checks
# ----------------------------
cad_df = pd.read_csv(INPUT_CAD_STRICT)
cvd_df = pd.read_csv(INPUT_CVD)

print(f"Loaded CAD strict rows: {len(cad_df)}")
print(f"Loaded CVD rows: {len(cvd_df)}")

required_common = {"SUBJECT_ID", "AGE", "GENDER"}
missing_cad_common = sorted(required_common - set(cad_df.columns))
missing_cvd_common = sorted(required_common - set(cvd_df.columns))

if missing_cad_common:
    raise ValueError(f"CAD strict file missing required columns: {missing_cad_common}")
if missing_cvd_common:
    raise ValueError(f"CVD file missing required columns: {missing_cvd_common}")

if "label" not in cad_df.columns:
    raise ValueError("CAD strict file must contain 'label' from Step 3.")
if "CVD_LABEL" not in cvd_df.columns:
    raise ValueError("CVD file must contain 'CVD_LABEL' from Step 3.")

cad_before = len(cad_df)
cad_df = cad_df[cad_df["label"].isin([0, 1])].copy()
print(f"Dropped CAD unresolved rows: {cad_before - len(cad_df)}")

print("CAD strict label distribution (row-level):")
print(cad_df["label"].value_counts(dropna=False))

print("CVD label distribution (row-level):")
print(cvd_df["CVD_LABEL"].value_counts(dropna=False))

## **Load and Validate Labeled Data from Step 3**

Load CAD strict and CVD datasets. Remove unresolved rows and verify required columns.

In [ ]:
# ----------------------------
# Helper functions
# ----------------------------
def normalize_icd9(code):
    if pd.isna(code):
        return ""
    return str(code).strip().upper().replace(".", "")

def extract_codes_from_row(row):
    if "ICD9_CODES" in row and pd.notna(row.get("ICD9_CODES")):
        return [normalize_icd9(x) for x in str(row.get("ICD9_CODES")).split(";") if str(x).strip()]

    if "4140x_codes" in row and pd.notna(row.get("4140x_codes")):
        return [normalize_icd9(x) for x in str(row.get("4140x_codes")).split(";") if str(x).strip()]

    return []

def map_gender_to_numeric(series):
    gender_mapping = {
        "M": 1, "F": 0,
        "Male": 1, "Female": 0,
        "male": 1, "female": 0
    }
    return series.map(gender_mapping)

def ensure_comorbidity_flags(df_in):
    df_out = df_in.copy()

    if all(c in df_out.columns for c in ["is_diabetic", "has_high_cholesterol", "is_obese"]):
        for c in ["is_diabetic", "has_high_cholesterol", "is_obese"]:
            df_out[c] = pd.to_numeric(df_out[c], errors="coerce").fillna(0).astype(int)
        return df_out

    codes_series = df_out.apply(extract_codes_from_row, axis=1)
    df_out["is_diabetic"] = codes_series.apply(lambda cs: int(any(c.startswith("250") for c in cs)))
    df_out["has_high_cholesterol"] = codes_series.apply(lambda cs: int(any(c.startswith("272") for c in cs)))
    df_out["is_obese"] = codes_series.apply(lambda cs: int(any(c.startswith("278") for c in cs)))
    return df_out

def preprocess_common_features(df_in):
    out = df_in.copy()

    out["GENDER_NUM"] = map_gender_to_numeric(out["GENDER"] if "GENDER" in out.columns else pd.Series(index=out.index, dtype=object))

    if "AGE" in out.columns and out["AGE"].isna().any():
        age_median = out["AGE"].median()
        out["AGE"] = out["AGE"].fillna(age_median)

    if out["GENDER_NUM"].isna().any():
        gender_mode = out["GENDER_NUM"].mode(dropna=True)
        fill_val = int(gender_mode.iloc[0]) if len(gender_mode) > 0 else 1
        out["GENDER_NUM"] = out["GENDER_NUM"].fillna(fill_val)

    out = ensure_comorbidity_flags(out)
    return out

def make_patient_level(df_in):
    sort_cols = [c for c in ["record_id"] if c in df_in.columns]
    if len(sort_cols) == 0:
        return df_in.drop_duplicates(subset=["SUBJECT_ID"], keep="first").copy()
    return df_in.sort_values(sort_cols).drop_duplicates(subset=["SUBJECT_ID"], keep="first").copy()

## **Define Feature Engineering Functions**

Helper functions to encode sex, clean age/comorbidity fields, and prepare patient-level features.

In [ ]:
# ----------------------------
# Feature preprocessing
# ----------------------------
cad_work = preprocess_common_features(cad_df)
cad_work["CAD_LABEL"] = cad_work["label"].astype(int)

cvd_work = preprocess_common_features(cvd_df)
cvd_work["CVD_LABEL"] = cvd_work["CVD_LABEL"].astype(int)

print("CAD strict preprocessing complete.")
print(cad_work[["CAD_LABEL", "AGE", "GENDER_NUM", "is_diabetic", "has_high_cholesterol", "is_obese"]].head())

print("CVD preprocessing complete.")
print(cvd_work[["CVD_LABEL", "AGE", "GENDER_NUM", "is_diabetic", "has_high_cholesterol", "is_obese"]].head())

## **Preprocess Clinical Features**

Apply sex encoding, age imputation, and comorbidity-field cleanup to both CAD and CVD datasets.

In [ ]:
# ----------------------------
# Build patient-level datasets
# ----------------------------
base_cols = ["SUBJECT_ID", "AGE", "GENDER", "GENDER_NUM"]
flag_cols = ["is_diabetic", "has_high_cholesterol", "is_obese"]
extra_cols = ["patient_id", "record_id", "group", "fs"]
ppg_cols_cad = sorted([c for c in cad_work.columns if c.startswith("ppg_min_")])
ppg_cols_cvd = sorted([c for c in cvd_work.columns if c.startswith("ppg_min_")])

cad_cols = [c for c in base_cols + ["CAD_LABEL"] + flag_cols + extra_cols + ppg_cols_cad if c in cad_work.columns]
cvd_cols = [c for c in base_cols + ["CVD_LABEL"] + flag_cols + extra_cols + ppg_cols_cvd if c in cvd_work.columns]

cad_patient = make_patient_level(cad_work[cad_cols].copy())
cvd_patient = make_patient_level(cvd_work[cvd_cols].copy())

cad_patient_case = cad_patient[cad_patient["CAD_LABEL"] == 1].copy()
cad_patient_control = cad_patient[cad_patient["CAD_LABEL"] == 0].copy()

cvd_patient_case = cvd_patient[cvd_patient["CVD_LABEL"] == 1].copy()
cvd_patient_control = cvd_patient[cvd_patient["CVD_LABEL"] == 0].copy()

print("Built patient-level datasets.")
print(f"CAD strict patient-level rows: {len(cad_patient)}")
print(f"CVD patient-level rows: {len(cvd_patient)}")
print(f"CAD PPG feature columns included: {len(ppg_cols_cad)}")
print(f"CVD PPG feature columns included: {len(ppg_cols_cvd)}")

## **Aggregate to Patient-Level Datasets**

Group waveform-level data by `SUBJECT_ID` to create one retained record per patient.

In [ ]:
# ----------------------------
# Save patient-level outputs
# ----------------------------
cad_patient.to_csv(OUTPUT_CAD_PATIENT_FULL, index=False)
cad_patient_case.to_csv(OUTPUT_CAD_PATIENT_CASE, index=False)
cad_patient_control.to_csv(OUTPUT_CAD_PATIENT_CONTROL, index=False)

cvd_patient.to_csv(OUTPUT_CVD_PATIENT_FULL, index=False)
cvd_patient_case.to_csv(OUTPUT_CVD_PATIENT_CASE, index=False)
cvd_patient_control.to_csv(OUTPUT_CVD_PATIENT_CONTROL, index=False)

print("Saved CAD strict patient-level outputs:")
print(f"  Full: {OUTPUT_CAD_PATIENT_FULL} ({len(cad_patient)} rows)")
print(f"  CAD only: {OUTPUT_CAD_PATIENT_CASE} ({len(cad_patient_case)} rows)")
print(f"  Non-CAD only: {OUTPUT_CAD_PATIENT_CONTROL} ({len(cad_patient_control)} rows)")

print("Saved CVD patient-level outputs:")
print(f"  Full: {OUTPUT_CVD_PATIENT_FULL} ({len(cvd_patient)} rows)")
print(f"  CVD only: {OUTPUT_CVD_PATIENT_CASE} ({len(cvd_patient_case)} rows)")
print(f"  Non-CVD only: {OUTPUT_CVD_PATIENT_CONTROL} ({len(cvd_patient_control)} rows)")

## **Export Final Patient-Level Datasets**

Save separate CSV files for CAD vs Non-CAD and CVD vs Non-CVD classification tasks.

In [ ]:
# ----------------------------
# Final QC summary
# ----------------------------
print("\n=== QC SUMMARY ===")
print("CAD strict patient-level distribution:")
print(cad_patient["CAD_LABEL"].value_counts(dropna=False))

print("CVD patient-level distribution:")
print(cvd_patient["CVD_LABEL"].value_counts(dropna=False))

print("CAD PPG columns included:")
print(ppg_cols_cad)

print("CVD PPG columns included:")
print(ppg_cols_cvd)

print("Missing values (CAD strict patient-level):")
cad_qc_cols = [c for c in ["AGE", "GENDER", "GENDER_NUM", "CAD_LABEL", "is_diabetic", "has_high_cholesterol", "is_obese"] if c in cad_patient.columns]
print(cad_patient[cad_qc_cols].isna().sum())

print("Missing values (CVD patient-level):")
cvd_qc_cols = [c for c in ["AGE", "GENDER", "GENDER_NUM", "CVD_LABEL", "is_diabetic", "has_high_cholesterol", "is_obese"] if c in cvd_patient.columns]
print(cvd_patient[cvd_qc_cols].isna().sum())

## **Quality Control Summary**

Validate label distributions, feature completeness, and missing value statistics.

In [ ]:
# ----------------------------
# Preview generated datasets
# ----------------------------
preview_files = [
    ("CAD vs Non-CAD (strict)", OUTPUT_CAD_PATIENT_FULL),
    ("CAD only (strict)", OUTPUT_CAD_PATIENT_CASE),
    ("Non-CAD only (strict)", OUTPUT_CAD_PATIENT_CONTROL),
    ("CVD vs Non-CVD", OUTPUT_CVD_PATIENT_FULL),
    ("CVD only", OUTPUT_CVD_PATIENT_CASE),
    ("Non-CVD only", OUTPUT_CVD_PATIENT_CONTROL),
]

preview_base_cols = [
    "SUBJECT_ID", "AGE", "GENDER", "GENDER_NUM",
    "CAD_LABEL", "CVD_LABEL",
    "is_diabetic", "has_high_cholesterol", "is_obese",
    "ppg_min_3", "ppg_min_4", "ppg_min_5", "ppg_min_6", "ppg_min_7", "ppg_min_8"
]

for dataset_name, file_path in preview_files:
    preview_df = pd.read_csv(file_path)
    preview_cols = [c for c in preview_base_cols if c in preview_df.columns]
    print(f"\n=== {dataset_name} ===")
    print(f"Rows: {len(preview_df)} | Columns: {len(preview_df.columns)}")
    display(preview_df[preview_cols].head())

## **Preview Generated Datasets**

Display first rows from all generated patient-level datasets to verify structure and content.